In [1]:
print("hello")

hello


In [12]:
from unsloth import FastLanguageModel
import torch
import json
import random
import os

In [13]:
model_name = 'unsloth/Phi-3-mini-4k-instruct-bnb-4bit'

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.1.4: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 5070 Ti. Num GPUs = 1. Max memory: 15.447 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
FastLanguageModel.for_inference(model)

In [14]:
categories = {
    "Information Technology": "laptop won't turn on, wifi broken, server crash, blue screen, printer offline, install software, recover files, virus removal, coding help, password reset",
    "Cooking": "private chef needed, meal prep for week, catering for party, vegan menu, teaching how to cook, food delivery for event, baking lessons, dietary restrictions",
    "Handiworks": "assemble furniture, fix door handle, hang shelves, painting wall, repair fence, fix roof leak, drywall patch, carpentry, broken window, lock replacement",
    "Plumbing": "sink leaking, toilet clogged, low water pressure, pipe burst, install faucet, shower not draining, water heater broken, blocked drain, leak detection",
    "Electricity": "outlet not working, light switch broken, breaker keeps tripping, install ceiling fan, rewire house, flickering lights, install EV charger, fuse blown",
    "Cleaning": "deep clean apartment, carpet cleaning, window washing, move-out cleaning, office cleaning, power wash driveway, tidy up messy room, laundry service",
    "Education": "math tutor, piano lessons, learn spanish, sat prep, help with history homework, physics tutor, coding lessons, essay writing help, english teacher",
    "Well Being": "yoga instructor, meditation coach, life coaching, stress management, mindfulness session, spiritual guidance, relaxation techniques, personal mentor",
    "Health": "nursing care for elderly, physiotherapy, wound dressing, post-surgery care, check blood pressure, medical assistance, home nurse, injection service",
    "Accounting": "file taxes, bookkeeping for small business, audit assistance, financial planning, payroll help, quickbooks support, tax return, expense tracking"
}

In [ ]:
generation_prompt = """You are a synthetic data generator. Generate 60 unique, diverse, and realistic user job requests for the category: "{category}".

Context keywords to inspire you: {keywords}

Rules:
1. The requests must sound like REAL customers (some angry, some polite, some urgent, some short, some detailed).
2. Do NOT number the list.
3. Output ONLY the requests, one per line.
4. Do not start lines with "I need" every time. Vary the phrasing.
5. Do not use the category name in the request (e.g., don't say "I need plumbing", say "my pipe burst").
"""

In [16]:
data = []

print(f"Starting generation for {len(categories)} categories...")

for cat, keywords in categories.items():
    print(f"--> Generating data for: {cat}...")

    for i in range(2): #the request ask LLM to generate 50 query once, twice means 100
        messages = [
            {"role": "user", "content": generation_prompt.format(category=cat, keywords=keywords)}
        ]
        inputs = tokenizer.apply_chat_template(
            messages, 
            tokenize=True, 
            add_generation_prompt=True, 
            return_tensors="pt"
        ).to("cuda")

        outputs = model.generate(
            inputs, 
            max_new_tokens=2048,
            temperature=0.9, # High temperature = more creativity/variety
            use_cache=True
        )
        
        # Decode output
        decoded_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        
        # Extract the assistant's response (handling different Llama response formats)
        if "assistant" in decoded_text:
            raw_response = decoded_text.split("assistant")[-1].strip()
        else:
            raw_response = decoded_text
            
        lines = raw_response.split("\n")
        
        # Clean and filter lines
        count = 0
        for line in lines:
            line = line.strip()
            # Remove bullets, numbers, and garbage
            clean_line = line.lstrip("- ").lstrip("* ").lstrip("1234567890. ").strip()
            
            # Keep only valid-looking lines
            if len(clean_line) > 10 and not clean_line.startswith("Here is") and not clean_line.startswith("Sure"):
                data.append({"input": clean_line, "output": cat})
                count += 1
        
        print(f"    Batch {i+1}: Generated {count} examples")

# 5. SAVE TO FILE (JSONL format)
print(f"Shuffling and saving {len(data)} examples...")
random.shuffle(data)

Starting generation for 10 categories...
--> Generating data for: Information Technology...
    Batch 1: Generated 75 examples
    Batch 2: Generated 75 examples
--> Generating data for: Cooking...
    Batch 1: Generated 84 examples
    Batch 2: Generated 84 examples
--> Generating data for: Handiworks...
    Batch 1: Generated 117 examples
    Batch 2: Generated 117 examples
--> Generating data for: Plumbing...
    Batch 1: Generated 106 examples
    Batch 2: Generated 106 examples
--> Generating data for: Electricity...
    Batch 1: Generated 70 examples
    Batch 2: Generated 70 examples
--> Generating data for: Cleaning...
    Batch 1: Generated 27 examples
    Batch 2: Generated 27 examples
--> Generating data for: Education...
    Batch 1: Generated 124 examples
    Batch 2: Generated 124 examples
--> Generating data for: Well Being...
    Batch 1: Generated 94 examples
    Batch 2: Generated 94 examples
--> Generating data for: Health...
    Batch 1: Generated 6 examples
    Bat

In [17]:
with open("train2.jsonl", "w", encoding='utf-8') as f:
    for entry in data:
        json.dump(entry, f)
        f.write("\n")

print("SUCCESS! File 'train.jsonl' is ready.")

SUCCESS! File 'train.jsonl' is ready.


In [ ]:
import json
import re

input_file = "train2.jsonl"
output_file = "train2_c.jsonl"


with open(input_file, 'r') as f_in, open(output_file, 'w') as f_out:
    for line in f_in:
        try:
            data = json.loads(line)
            data['input'] = data['input'].replace('\"', '') #clean /"
            data['input'] = re.sub(r'0{5,}', '', data['input']) #clean repeat character 
            f_out.write(json.dumps(data) + '\n')
        except json.JSONDecodeError:
            continue

print("Done")

Done
